`python
# ================================================================
# NOTEBOOK 2: CHẠY BENCHMARK E5 — TTS VÀ SCHEDULING
# Môi trường: Google Colab | Runtime: T4 GPU (khuyên dùng)
# YÊU CẦU: Đã chạy xong Notebook 1
# ================================================================
`

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 1 — Mount Google Drive + Tải dữ liệu (CHẠY ĐẦU TIÊN)│
# └─────────────────────────────────────────────────────────────┘
# Colab sẽ hỏi xin quyền → Nhấn "Connect to Google Drive"

from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')

# ---- Đường dẫn Drive (phải khớp với Notebook 1) ----
DRIVE_DIR   = '/content/drive/MyDrive/E5_TTS_Benchmark'
LOCAL_DIR   = '/content/e5_tts_benchmark'
RESULTS_DIR = LOCAL_DIR + '/results'
AUDIO_DIR   = LOCAL_DIR + '/audio'

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

# ---- Tải benchmark_200.json từ Drive về local ----
drive_json = f'{DRIVE_DIR}/benchmark_200.json'
local_json = f'{LOCAL_DIR}/benchmark_200.json'

if not os.path.exists(drive_json):
    raise FileNotFoundError(
        f"❌ Không tìm thấy file trên Drive: {drive_json}\n"
        "   → Hãy chạy Notebook 1 trước!"
    )

shutil.copy(drive_json, local_json)

print("✅ Đã kết nối Google Drive!")
print(f"   📄 Tải về: {local_json}")
print(f"   📁 Drive : {DRIVE_DIR}")
print()
print("👉 Tiếp tục chạy CELL 2 →")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 2 — Cài đặt thư viện                                │
# └─────────────────────────────────────────────────────────────┘
# ⚠️ Chờ cell này hoàn tất (3-5 phút) rồi mới chạy tiếp

!pip install vieneu --quiet
!pip install transformers torch torchaudio --quiet
!pip install soundfile scipy numpy pandas tqdm --quiet

print("✅ Đã cài đặt xong tất cả thư viện!")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 3 — Import và kiểm tra môi trường                   │
# └─────────────────────────────────────────────────────────────┘

import json, time, re, math, logging, traceback
from typing import Iterator, List, Tuple, Dict, Any
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm import tqdm

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
logger = logging.getLogger("E5")

BENCHMARK_FILE = Path(local_json)
TARGET_SR = 24000   # 24 kHz PCM mono theo E5
WARMUP_N  = 10      # Bỏ qua 10 mẫu đầu theo E0

# Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"⚙️  Device: {device.upper()}")
if torch.cuda.is_available():
    logger.info(f"   GPU : {torch.cuda.get_device_name(0)}")
    logger.info(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    logger.info("   CPU only — TTS vẫn chạy được, chậm hơn GPU")

# Tải dữ liệu
with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

logger.info(f"✅ Tải {len(benchmark_data)} câu từ benchmark_200.json")
logger.info(f"   HUIT : {sum(1 for x in benchmark_data if 'HUIT' in x['source'])}")
logger.info(f"   VSFC : {sum(1 for x in benchmark_data if 'VSFC' in x['source'])}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 4 — Text Normalization (dùng chung cả 2 backend)    │
# └─────────────────────────────────────────────────────────────┘

def normalize_text(text: str) -> str:
    text = text.strip()
    text = text.replace("``", '"').replace("''", '"')
    text = text.replace("–", "-").replace("—", ",")
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    return text.strip()

print("✅ normalize_text() OK")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 5 — Hàm Resample tiện ích                           │
# └─────────────────────────────────────────────────────────────┘

def resample(audio: np.ndarray, src_sr: int, tgt_sr: int) -> np.ndarray:
    """Resample về target sample rate, chuyển sang mono nếu stereo."""
    import torchaudio.functional as F
    if audio.ndim > 1:
        audio = audio.mean(axis=0)
    if src_sr == tgt_sr:
        return audio
    tensor = torch.from_numpy(audio.astype(np.float32)).unsqueeze(0)
    resampled = F.resample(tensor, src_sr, tgt_sr)
    return resampled.squeeze(0).numpy()

print("✅ resample() OK")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 6 — Backend 1: VieNeu-TTS-v3-Turbo                  │
# └─────────────────────────────────────────────────────────────┘

class VieNeuTTSBackend:
    """pnnbao-ump/VieNeu-TTS-v3-Turbo — ONNX CPU, output 48kHz → 24kHz"""
    
    SOURCE_SR = 48000

    def __init__(self, voice="Mai Anh"):
        logger.info("⏳ Đang tải VieNeu-TTS-v3-Turbo...")
        from vieneu import Vieneu
        self.model = Vieneu()
        self.voice = voice
        logger.info(f"✅ VieNeu-TTS ready | Voice: {voice}")

    def synthesize_full(self, text: str) -> Tuple[np.ndarray, int]:
        audio = self.model.infer(text=text, voice=self.voice)
        audio_np = np.array(audio, dtype=np.float32)
        return resample(audio_np, self.SOURCE_SR, TARGET_SR), TARGET_SR

    def synthesize_stream(self, text: str) -> Iterator[Tuple[np.ndarray, int]]:
        try:
            for chunk in self.model.infer_stream(text=text):
                chunk_np = np.array(chunk, dtype=np.float32)
                yield resample(chunk_np, self.SOURCE_SR, TARGET_SR), TARGET_SR
        except AttributeError:
            # Fallback: chia nhỏ audio thành chunk ~1 giây
            audio, sr = self.synthesize_full(text)
            for i in range(0, len(audio), sr):
                yield audio[i:i+sr], sr


logger.info("🚀 Khởi tạo Backend 1: VieNeu-TTS-v3-Turbo...")
try:
    backend_vieneu = VieNeuTTSBackend(voice="Mai Anh")
    BACKEND1_OK = True
except Exception as e:
    logger.error(f"❌ VieNeu-TTS lỗi: {e}")
    BACKEND1_OK = False

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 7 — Backend 2: facebook/mms-tts-vie                  │
# └─────────────────────────────────────────────────────────────┘

class MMSTTSBackend:
    """facebook/mms-tts-vie — CPU, output 16kHz → 24kHz"""
    
    MODEL_ID  = "facebook/mms-tts-vie"
    SOURCE_SR = 16000

    def __init__(self):
        logger.info("⏳ Đang tải facebook/mms-tts-vie...")
        from transformers import VitsModel, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_ID)
        self.model = VitsModel.from_pretrained(self.MODEL_ID).eval().to("cpu")
        logger.info("✅ MMS-TTS-VIE ready | CPU")

    def _generate(self, text: str) -> np.ndarray:
        inputs = self.tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            output = self.model(**inputs).waveform
        audio_np = output.squeeze(0).cpu().numpy().astype(np.float32)
        return resample(audio_np, self.SOURCE_SR, TARGET_SR)

    def synthesize_full(self, text: str) -> Tuple[np.ndarray, int]:
        return self._generate(text), TARGET_SR

    def synthesize_stream(self, text: str) -> Iterator[Tuple[np.ndarray, int]]:
        audio, sr = self.synthesize_full(text)
        chunk_size = sr // 2  # ~0.5 giây mỗi chunk
        for i in range(0, len(audio), chunk_size):
            yield audio[i:i+chunk_size], sr


logger.info("🚀 Khởi tạo Backend 2: facebook/mms-tts-vie...")
try:
    backend_mms = MMSTTSBackend()
    BACKEND2_OK = True
except Exception as e:
    logger.error(f"❌ MMS-TTS lỗi: {e}")
    BACKEND2_OK = False

logger.info(f"\n{'='*40}")
logger.info(f"Backend 1 VieNeu : {'✅ OK' if BACKEND1_OK else '❌ LỖI'}")
logger.info(f"Backend 2 MMS    : {'✅ OK' if BACKEND2_OK else '❌ LỖI'}")
logger.info(f"{'='*40}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 8 — Ba chính sách Scheduling P0 / P1 / P2           │
# └─────────────────────────────────────────────────────────────┘

def policy_p0(text: str, backend=None) -> List[str]:
    """P0: Chờ toàn bộ text → 1 lần gọi TTS duy nhất."""
    return [text]

def policy_p1(text: str, backend=None) -> List[str]:
    """P1: Flush ngay khi gặp dấu [ . ? ! ; : ]"""
    chunks = re.split(r'(?<=[.?!;:])\s+', text.strip())
    chunks = [c.strip() for c in chunks if c.strip()]
    return chunks or [text]

def policy_p2(text: str, backend=None,
              min_chars: int = 60, force_flush: int = 120) -> List[str]:
    """
    P2: Buffer token stream.
    - Flush tại dấu câu khi buffer ≥ 60 ký tự
    - Ép flush khi buffer ≥ 120 ký tự
    """
    PUNCT  = {'.', '?', '!', ';', ':'}
    buffer = ""
    chunks = []
    for token in text.split(' '):
        buffer += token + ' '
        last = token.rstrip()[-1] if token.rstrip() else ''
        if len(buffer) >= force_flush:
            chunks.append(buffer.strip()); buffer = ""
        elif last in PUNCT and len(buffer) >= min_chars:
            chunks.append(buffer.strip()); buffer = ""
    if buffer.strip():
        chunks.append(buffer.strip())
    return chunks or [text]

# Kiểm tra
sample = "Học phí học kỳ 1 được thu theo tín chỉ. Mức thu là 550.000 đồng mỗi tín chỉ. Sinh viên cần đóng trước ngày 30."
print(f"P0: {policy_p0(sample)}")
print(f"P1: {policy_p1(sample)}")
print(f"P2: {policy_p2(sample)}")
print("✅ 3 policy OK")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 9 — Hàm đo Metrics                                  │
# └─────────────────────────────────────────────────────────────┘

def duration_s(audio: np.ndarray, sr: int) -> float:
    return len(audio) / sr

def compute_rtf(render_t: float, audio_dur: float) -> float:
    return render_t / audio_dur if audio_dur > 0 else float('inf')

def detect_underrun(render_times: List[float], durations: List[float]) -> bool:
    cum_play = cum_render = 0.0
    for rt, dur in zip(render_times, durations):
        cum_render += rt
        if cum_render > cum_play and cum_play > 0:
            return True
        cum_play += dur
    return False

def bootstrap_ci95(vals: List[float], n=1000) -> Tuple[float, float]:
    if not vals:
        return (float('nan'), float('nan'))
    arr = np.array(vals)
    boots = [np.median(np.random.choice(arr, len(arr), replace=True)) for _ in range(n)]
    return (np.percentile(boots, 2.5), np.percentile(boots, 97.5))

print("✅ Hàm metrics OK")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 10 — Hàm chạy 1 trial                               │
# └─────────────────────────────────────────────────────────────┘

def run_trial(item, backend_name, backend, policy_name, policy_fn,
              save_audio=False) -> Dict[str, Any]:
    qid  = item["question_id"]
    text = normalize_text(item["text"])
    res  = {"question_id": qid, "source": item["source"],
            "backend": backend_name, "policy": policy_name,
            "text": text, "char_len": len(text)}
    try:
        text_chunks = policy_fn(text, backend)
        t_start = time.perf_counter()
        all_audio, render_times, durations = [], [], []
        ttfa = None

        for chunk_text in text_chunks:
            if not chunk_text.strip():
                continue
            t0 = time.perf_counter()

            if policy_name == "P2":
                gen = list(backend.synthesize_stream(chunk_text))
                for a, sr in gen:
                    if ttfa is None:
                        ttfa = time.perf_counter() - t_start
                    all_audio.append(a)
                    durations.append(duration_s(a, sr))
            else:
                audio, sr = backend.synthesize_full(chunk_text)
                if ttfa is None:
                    ttfa = time.perf_counter() - t_start
                all_audio.append(audio)
                durations.append(duration_s(audio, sr))

            render_times.append(time.perf_counter() - t0)

        total_lat  = time.perf_counter() - t_start
        full_audio = np.concatenate(all_audio) if all_audio else np.array([], np.float32)
        total_dur  = sum(durations)

        if save_audio and len(full_audio) > 0:
            apath = f"{AUDIO_DIR}/{qid}_{backend_name}_{policy_name}.wav"
            sf.write(apath, full_audio, TARGET_SR, subtype="PCM_16")
            res["audio_path"] = apath

        res.update({
            "ttfa_s"          : round(ttfa, 4) if ttfa else None,
            "total_latency_s" : round(total_lat, 4),
            "total_audio_dur_s": round(total_dur, 4),
            "rtf"             : round(compute_rtf(sum(render_times), total_dur), 4),
            "has_underrun"    : int(detect_underrun(render_times, durations)),
            "n_chunks"        : len(text_chunks),
            "error"           : None,
        })
    except Exception as e:
        res.update({"ttfa_s": None, "total_latency_s": None,
                    "rtf": None, "has_underrun": None,
                    "n_chunks": None, "error": str(e)})
        logger.warning(f"⚠️ Lỗi [{backend_name}|{policy_name}|{qid}]: {e}")
    return res

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 11 — 🚀 CHẠY BENCHMARK (2 × 3 × 200 trial)          │
# └─────────────────────────────────────────────────────────────┘
# ⏱️  T4 GPU: ~30-60 phút | CPU: ~2-4 giờ
# Kết quả ghi từng dòng vào JSONL → an toàn nếu bị ngắt

BACKENDS = {}
if BACKEND1_OK: BACKENDS["VieNeu-v3-Turbo"] = backend_vieneu
if BACKEND2_OK: BACKENDS["MMS-TTS-VIE"]     = backend_mms

POLICIES = {"P0": policy_p0, "P1": policy_p1, "P2": policy_p2}

RESULTS_FILE = Path(RESULTS_DIR) / "e5_raw_results.jsonl"
total_trials = len(BACKENDS) * len(POLICIES) * len(benchmark_data)

logger.info(f"\n{'='*55}")
logger.info(f"🚀 BẮT ĐẦU BENCHMARK E5")
logger.info(f"   Backends  : {list(BACKENDS.keys())}")
logger.info(f"   Policies  : {list(POLICIES.keys())}")
logger.info(f"   Câu test  : {len(benchmark_data)}")
logger.info(f"   Tổng trial: {total_trials}")
logger.info(f"{'='*55}")

all_results = []

with open(RESULTS_FILE, "w", encoding="utf-8") as log_f:
    for bk_name, backend in BACKENDS.items():
        for pol_name, pol_fn in POLICIES.items():
            tag = f"[{bk_name} | {pol_name}]"
            logger.info(f"\n--- {tag} ---")
            
            # Warm-up (không tính vào số liệu theo E0)
            logger.info(f"   Warm-up {WARMUP_N} mẫu...")
            for w in benchmark_data[:WARMUP_N]:
                try: run_trial(w, bk_name, backend, pol_name, pol_fn)
                except: pass
            
            # Chạy thật
            for i, item in enumerate(tqdm(benchmark_data, desc=tag, unit="câu")):
                res = run_trial(item, bk_name, backend, pol_name, pol_fn,
                                save_audio=(i < 10))
                all_results.append(res)
                log_f.write(json.dumps(res, ensure_ascii=False) + "\n")
                log_f.flush()
            
            # Log tóm tắt sau mỗi combo
            ok = [r for r in all_results
                  if r["backend"]==bk_name and r["policy"]==pol_name
                  and r.get("ttfa_s") is not None]
            if ok:
                ttfa_v = [r["ttfa_s"] for r in ok]
                rtf_v  = [r["rtf"]    for r in ok]
                ur     = sum(r["has_underrun"] for r in ok)
                logger.info(
                    f"   ✅ {tag} | "
                    f"TTFA p50={np.percentile(ttfa_v,50):.3f}s "
                    f"p95={np.percentile(ttfa_v,95):.3f}s | "
                    f"RTF p50={np.percentile(rtf_v,50):.3f} | "
                    f"Underrun={ur}/{len(ok)}"
                )

logger.info(f"\n✅ Xong! {len(all_results)}/{total_trials} trial")
logger.info(f"   Raw JSONL: {RESULTS_FILE}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 12 — In Bảng 6 kết quả                              │
# └─────────────────────────────────────────────────────────────┘

df_raw = pd.DataFrame(all_results)
df_ok  = df_raw[df_raw["error"].isna()].copy()
df_err = df_raw[df_raw["error"].notna()]

if len(df_err): print(f"⚠️  {len(df_err)} trial lỗi — kiểm tra log")

print(f"\n{'='*70}")
print(f"📊 BẢNG 6 — E5: TTS VÀ SCHEDULING")
print(f"{'='*70}")

rows = []
for bk in df_ok["backend"].unique():
    for pol in ["P0", "P1", "P2"]:
        sub = df_ok[(df_ok["backend"]==bk) & (df_ok["policy"]==pol)]
        if sub.empty: continue
        ttfa = sub["ttfa_s"].dropna().tolist()
        rtf  = sub["rtf"].dropna().tolist()
        lat  = sub["total_latency_s"].dropna().tolist()
        ur   = sub["has_underrun"].mean()*100
        ci_t = bootstrap_ci95(ttfa)
        ci_r = bootstrap_ci95(rtf)
        rows.append({
            "Backend"          : bk,
            "Policy"           : pol,
            "N"                : len(sub),
            "TTFA p50 (s)"     : round(np.percentile(ttfa,50),3) if ttfa else "—",
            "TTFA p95 (s)"     : round(np.percentile(ttfa,95),3) if ttfa else "—",
            "TTFA CI95"        : f"[{ci_t[0]:.3f},{ci_t[1]:.3f}]" if ttfa else "—",
            "RTF p50"          : round(np.percentile(rtf,50),3) if rtf else "—",
            "Underrun (%)"     : round(ur,1),
            "Total lat p50 (s)": round(np.percentile(lat,50),3) if lat else "—",
        })

df_table = pd.DataFrame(rows)
print(df_table.to_string(index=False))
print(f"{'='*70}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 13 — Vẽ biểu đồ so sánh                             │
# └─────────────────────────────────────────────────────────────┘

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("E5 — TTS & Scheduling: 2 Backend × 3 Policy", fontsize=14, fontweight='bold')

COLORS = {"P0": "#4C72B0", "P1": "#DD8452", "P2": "#55A868"}
BK_LIST = list(df_ok["backend"].unique())
x = np.arange(len(BK_LIST))
w = 0.25

for ax_i, (metric, ylabel, title) in enumerate([
    ("ttfa_s",       "Giây (s)", "TTFA p50"),
    ("rtf",          "RTF",      "RTF p50"),
    ("has_underrun", "Tỉ lệ (%)", "Underrun (%)"),
]):
    ax = axes[ax_i]
    for p_i, pol in enumerate(["P0","P1","P2"]):
        vals = []
        for bk in BK_LIST:
            s = df_ok[(df_ok["backend"]==bk)&(df_ok["policy"]==pol)][metric].dropna()
            if metric == "has_underrun":
                vals.append(s.mean()*100 if len(s)>0 else 0)
            else:
                vals.append(np.percentile(s,50) if len(s)>0 else 0)
        ax.bar(x + p_i*w, vals, w, label=pol, color=COLORS[pol], alpha=0.85, edgecolor="black")
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.set_xticks(x+w); ax.set_xticklabels([b.replace("-","\n") for b in BK_LIST], fontsize=9)
    ax.legend(title="Policy")
    if metric=="rtf": ax.axhline(1.0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
chart_path = Path(RESULTS_DIR) / "e5_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Biểu đồ lưu tại: {chart_path}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 14 — Lưu tất cả kết quả lên Google Drive            │
# └─────────────────────────────────────────────────────────────┘

import shutil

# Lưu CSV đầy đủ
full_csv = Path(RESULTS_DIR) / "e5_full_results.csv"
df_raw.to_csv(full_csv, index=False, encoding="utf-8-sig")

# Lưu bảng 6
table_csv = Path(RESULTS_DIR) / "table6_e5.csv"
df_table.to_csv(table_csv, index=False, encoding="utf-8-sig")

# ---- Sao chép toàn bộ thư mục results lên Drive ----
drive_results = Path(DRIVE_DIR) / "results"
drive_results.mkdir(parents=True, exist_ok=True)

for f in Path(RESULTS_DIR).glob("*"):
    shutil.copy(str(f), str(drive_results / f.name))

# Sao chép một số audio mẫu lên Drive
drive_audio = Path(DRIVE_DIR) / "audio_samples"
drive_audio.mkdir(parents=True, exist_ok=True)
for f in list(Path(AUDIO_DIR).glob("*.wav"))[:20]:  # Lưu 20 file đầu
    shutil.copy(str(f), str(drive_audio / f.name))

print("\n✅ Đã lưu tất cả lên Google Drive!")
print(f"   📁 Drive: {DRIVE_DIR}/")
print(f"   ├── results/")
print(f"   │   ├── e5_raw_results.jsonl")
print(f"   │   ├── e5_full_results.csv")
print(f"   │   ├── table6_e5.csv")
print(f"   │   └── e5_chart.png")
print(f"   └── audio_samples/  (20 file WAV mẫu)")
print()
print("✅ HOÀN THÀNH THỰC NGHIỆM E5!")
print("👉 Bước tiếp theo: Nghe mù 20-30 người đánh giá Naturalness/Continuity/Intelligibility")